# Chapter 10 · Experimental and Quasi-Experimental Designs in Health: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the chapter's code on the case and check the numbers. All data are **synthetic**; no real patients appear.

**The case.** A university hypertension service with ten outpatient clinics decides whether to offer two antihypertensive drugs as a single pill instead of separate tablets; the outcome is uncontrolled blood pressure at 12 weeks. One table, `data/clinic_patients.csv`, has one row per adult over 24 months: months 1-6 everyone on separate tablets; months 7-12 an individually randomized trial; from month 13 the clinics adopt the single pill in two steps (five clinics in month 13, five in month 19).

**Part A** runs the chapter's code section by section. **Part B** gives the code and result for the chapter exercises that need code (statements are in the book). Run the cells in order.

## Part A. The case

### The comparison and the outcome risk

In [1]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
events = trial["uncontrolled_12w"].eq("yes").sum()
print(f"Patients: {len(trial)}; uncontrolled: {events}")
print(f"Overall risk: {events / len(trial):.4f}")

Patients: 1716; uncontrolled: 478
Overall risk: 0.2786


### Randomization and baseline balance

In [2]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
for arm, group in trial.groupby("allocated"):
    high = group["baseline_sbp"].ge(160).sum()
    print(f"{arm}: {high}/{len(group)} = {100*high/len(group):.1f}%")

separate tablets: 331/859 = 38.5%
single pill: 341/857 = 39.8%


### Intention to treat and uncertainty

In [3]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
counts = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
risks = counts["yes"] / counts.sum(axis=1)
p, q = risks.loc["single pill"], risks.loc["separate tablets"]
for arm in counts.index:
    print(f"{arm}: {counts.loc[arm, 'yes']}/{counts.loc[arm].sum()}")
print(f"Risk difference: {100*(p-q):.1f} percentage points")
print(f"Risk ratio: {p/q:.2f}")

separate tablets: 279/859
single pill: 199/857
Risk difference: -9.3 percentage points
Risk ratio: 0.71


In [4]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
x = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["separate tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["separate tablets"].sum()
p, q = a/n, b/m
z = NormalDist().inv_cdf(0.975)
d = 100*(p-q)
se = 100*sqrt(p*(1-p)/n + q*(1-q)/m)
sl = sqrt(1/a-1/n+1/b-1/m)
print(f"SE: {se:.4f}; normal multiplier: {z:.4f}")
print(f"Difference: {d:.1f}; 95% CI {d-z*se:.1f} to {d+z*se:.1f}")
print(f"Ratio: {p/q:.2f}; 95% CI "
      f"{exp(log(p/q)-z*sl):.2f} to {exp(log(p/q)+z*sl):.2f}")

SE: 2.1525; normal multiplier: 1.9600
Difference: -9.3; 95% CI -13.5 to -5.0
Ratio: 0.71; 95% CI 0.61 to 0.84


### Per-protocol questions and treatment received

In [5]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
same = trial["allocated"].eq("single pill") == trial["received"].eq("yes")
x = pd.crosstab(trial.loc[same, "allocated"],
                trial.loc[same, "uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["separate tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["separate tablets"].sum()
rr = (a/n)/(b/m)
se = sqrt(1/a-1/n+1/b-1/m)
z = NormalDist().inv_cdf(0.975)
print(f"Retained: {n+m}; events/total: {a}/{n}, {b}/{m}")
print(f"Restricted ratio: {rr:.2f}; 95% CI "
      f"{exp(log(rr)-z*se):.2f} to {exp(log(rr)+z*se):.2f}")

Retained: 1610; events/total: 169/782, 270/828
Restricted ratio: 0.66; 95% CI 0.56 to 0.78


### Cluster-randomized trials

In [6]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
rollout = records[records["month"].between(13, 18)]
for arm, group in rollout.groupby("allocated"):
    events = group["uncontrolled_12w"].eq("yes").sum()
    print(f"{arm}: {group['clinic'].nunique()} clinics; {events}/{len(group)}")
x = pd.crosstab(rollout["allocated"], rollout["uncontrolled_12w"])
r = x["yes"] / x.sum(axis=1)
print(f"Pooled ratio: {r.loc['single pill']/r.loc['separate tablets']:.2f}")

separate tablets: 5 clinics; 236/852
single pill: 5 clinics; 179/819
Pooled ratio: 0.79


In [7]:
import pandas as pd
from math import sqrt
from scipy.stats import t
records = pd.read_csv("data/clinic_patients.csv")
rollout = records[records["month"].between(13, 18)].copy()
rollout["y"] = rollout["uncontrolled_12w"].eq("yes")
r = rollout.groupby(["allocated", "clinic"])["y"].mean()
p, q = r.loc["single pill"], r.loc["separate tablets"]
u, v = p.var()/len(p), q.var()/len(q)
df = (u+v)**2 / (u*u/(len(p)-1) + v*v/(len(q)-1))
d, se = 100*(p.mean()-q.mean()), 100*sqrt(u+v)
cut = t.ppf(0.975, df)
print(f"Mean clinic risks: {p.mean():.4f}, {q.mean():.4f}")
print(f"SE: {se:.4f}; df: {df:.2f}; t cutoff: {cut:.4f}")
print(f"Difference: {d:.1f}; 95% CI {d-cut*se:.1f} to {d+cut*se:.1f}")

Mean clinic risks: 0.2176, 0.2776
SE: 1.6944; df: 8.00; t cutoff: 2.3061
Difference: -6.0; 95% CI -9.9 to -2.1


### Stepped-wedge cluster trials

In [8]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
records["y"] = records["uncontrolled_12w"].eq("yes")
def risk(switch_month, first, last):
    rows = records[records["clinic_start_month"].eq(switch_month)
                   & records["month"].between(first, last)]
    return rows["y"].mean()
late = risk(19, 19, 24) - risk(19, 13, 18)
early = risk(13, 19, 24) - risk(13, 13, 18)
print(f"Switched in month 13: {100*risk(13, 13, 18):.1f}%, {100*risk(13, 19, 24):.1f}%")
print(f"Switched in month 19: {100*risk(19, 13, 18):.1f}%, {100*risk(19, 19, 24):.1f}%")
print(f"Changes: {100*late:.1f}, {100*early:.1f} points")
print(f"Stepped-wedge contrast: {100*(late-early):.1f} points")

Switched in month 13: 21.9%, 19.5%
Switched in month 19: 27.7%, 21.0%
Changes: -6.7, -2.4 points
Stepped-wedge contrast: -4.3 points


### Before-and-after comparisons and secular trends

In [9]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
adopters = records[records["clinic_start_month"].eq(13)]
risks = []
for name, rows in [("before", adopters[adopters["month"].between(1, 6)]),
                   ("after", adopters[adopters["month"].between(13, 18)])]:
    a, n = rows["uncontrolled_12w"].eq("yes").sum(), len(rows)
    risks.append(a/n)
    print(f"{name}: {a}/{n} = {100*a/n:.1f}%")
print(f"After/before: {risks[1]/risks[0]:.2f}")

before: 311/830 = 37.5%
after: 179/819 = 21.9%
After/before: 0.58


### Controlled comparisons and difference in differences

In [10]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
def risk_in(rows):
    return rows["uncontrolled_12w"].eq("yes").mean()
adopt = records[records["clinic_start_month"].eq(13)]
comp = records[records["clinic_start_month"].eq(19)]
a0, a1 = risk_in(adopt[adopt["month"].between(1, 6)]), risk_in(adopt[adopt["month"].between(13, 18)])
b0, b1 = risk_in(comp[comp["month"].between(1, 6)]), risk_in(comp[comp["month"].between(13, 18)])
print(f"Adopting clinics: {100*a0:.1f}%, {100*a1:.1f}%")
print(f"Comparison clinics: {100*b0:.1f}%, {100*b1:.1f}%")
print(f"Changes: {100*(a1-a0):.1f}, {100*(b1-b0):.1f} points")
print(f"Difference in differences: {100*((a1-a0)-(b1-b0)):.1f} points")
print(f"Assumed untreated endpoint: {100*(a0+b1-b0):.1f}%")
print(f"Ratio of ratios: {(a1/a0)/(b1/b0):.2f}")

Adopting clinics: 37.5%, 21.9%
Comparison clinics: 35.7%, 27.7%
Changes: -15.6, -8.0 points
Difference in differences: -7.6 points
Assumed untreated endpoint: 29.5%
Ratio of ratios: 0.75


### Interrupted time series

In [11]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
adopters = records[records["clinic_start_month"].eq(13)]
monthly = adopters.groupby("month")["uncontrolled_12w"].agg(
    patients="size", uncontrolled=lambda s: s.eq("yes").sum())
first = monthly.iloc[0]
print(f"Before months: {(monthly.index < 7).sum()}")
print(f"Trial months: {monthly.index.to_series().between(7, 12).sum()}")
print(f"After months: {(monthly.index >= 13).sum()}")
print(f"First risk: {first['uncontrolled']}/{first['patients']}"
      f" = {100*first['uncontrolled']/first['patients']:.1f}%")

Before months: 6
Trial months: 6
After months: 12
First risk: 54/134 = 40.3%


## Part B. Exercise solutions

Exercises without code (conceptual answers) are answered in the book.

### Exercise 1. Baseline balance

In [12]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
for arm, group in trial.groupby("allocated"):
    high = group["baseline_sbp"].ge(160).sum()
    print(f"{arm}: n={len(group)}, age={group['age_years'].mean():.1f}, "
          f"high={high}, share={100*high/len(group):.1f}%")

separate tablets: n=859, age=60.3, high=331, share=38.5%
single pill: n=857, age=60.4, high=341, share=39.8%


### Exercise 3. Assignment and adherence questions

In [13]:
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
records = pd.read_csv("data/clinic_patients.csv")
trial = records[records["period"].eq("trial")]
x = pd.crosstab(trial["allocated"], trial["uncontrolled_12w"])
a, b = x.loc["single pill", "yes"], x.loc["separate tablets", "yes"]
n, m = x.loc["single pill"].sum(), x.loc["separate tablets"].sum()
p, q = a/n, b/m
z = NormalDist().inv_cdf(0.975)
se = 100*sqrt(p*(1-p)/n + q*(1-q)/m)
sl = sqrt(1/a-1/n+1/b-1/m)
print(f"Risks: {p:.4f}, {q:.4f}")
print(f"Difference: {100*(p-q):.1f}; 95% CI {100*(p-q)-z*se:.1f} to {100*(p-q)+z*se:.1f}")
print(f"Ratio: {p/q:.2f}; 95% CI {exp(log(p/q)-z*sl):.2f} to {exp(log(p/q)+z*sl):.2f}")
same = trial["allocated"].eq("single pill") == trial["received"].eq("yes")
print(f"Receipt matches assignment in {same.sum()} of {len(trial)} adults")

Risks: 0.2322, 0.3248
Difference: -9.3; 95% CI -13.5 to -5.0
Ratio: 0.71; 95% CI 0.61 to 0.84
Receipt matches assignment in 1610 of 1716 adults


### Exercise 7. Clinics within the allocated groups

In [14]:
import pandas as pd
from math import sqrt
from scipy.stats import t
records = pd.read_csv("data/clinic_patients.csv")
rollout = records[records["month"].between(13, 18)].copy()
rollout["y"] = rollout["uncontrolled_12w"].eq("yes")
table = rollout.groupby(["allocated", "clinic"])["y"].agg(["sum", "size"])
table["risk"] = 100 * table["sum"] / table["size"]
print(table.round(1).to_string())
r = rollout.groupby(["allocated", "clinic"])["y"].mean()
p, q = r.loc["single pill"], r.loc["separate tablets"]
u, v = p.var()/len(p), q.var()/len(q)
df = (u+v)**2 / (u*u/(len(p)-1) + v*v/(len(q)-1))
d, se = 100*(p.mean()-q.mean()), 100*sqrt(u+v)
cut = t.ppf(0.975, df)
print(f"Difference: {d:.1f}; 95% CI {d-cut*se:.1f} to {d+cut*se:.1f}")

                         sum  size  risk
allocated        clinic                 
separate tablets C01      50   167  29.9
                 C02      42   180  23.3
                 C05      50   167  29.9
                 C07      45   163  27.6
                 C10      49   175  28.0
single pill      C03      37   167  22.2
                 C04      35   171  20.5
                 C06      36   160  22.5
                 C08      27   148  18.2
                 C09      44   173  25.4
Difference: -6.0; 95% CI -9.9 to -2.1


### Exercise 9. The whole-service change

In [15]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
for name, rows in [("months 1-6", records[records["month"].between(1, 6)]),
                   ("months 19-24", records[records["month"].between(19, 24)])]:
    n = len(rows)
    a = rows["uncontrolled_12w"].eq("yes").sum()
    s = rows["received"].eq("yes").sum()
    print(f"{name}: uncontrolled {a}/{n} = {100*a/n:.1f}%; "
          f"uptake {s}/{n} = {100*s/n:.1f}%")

months 1-6: uncontrolled 621/1698 = 36.6%; uptake 56/1698 = 3.3%
months 19-24: uncontrolled 336/1659 = 20.3%; uptake 1527/1659 = 92.0%


### Exercise 10. The stepped-wedge contrast against the cluster comparison

In [16]:
import pandas as pd
records = pd.read_csv("data/clinic_patients.csv")
records["y"] = records["uncontrolled_12w"].eq("yes")
def risk(switch_month, first, last):
    rows = records[records["clinic_start_month"].eq(switch_month)
                   & records["month"].between(first, last)]
    return rows["y"].mean()
late = risk(19, 19, 24) - risk(19, 13, 18)
early = risk(13, 19, 24) - risk(13, 13, 18)
cluster = risk(13, 13, 18) - risk(19, 13, 18)
print(f"Stepped-wedge contrast: {100*(late-early):.1f} points")
print(f"Cluster comparison (pooled patients): {100*cluster:.1f} points")

Stepped-wedge contrast: -4.3 points
Cluster comparison (pooled patients): -5.8 points


---
*Synthetic data (seed 33, `generate_data.py`). Teaching assumptions only; nothing here is a clinical recommendation or an estimate for a real service.*